# fishi — OpenWorldSAM experiments

OpenWorldSAM uses a detectron2 + torch-2.5 stack that conflicts with `fishi[models]`,
so it runs in its **own** Colab GPU runtime. It writes per-cell metrics into the same
`metrics/` folder on Drive as the Grounded SAM notebook, so no merge step is needed.

In [ ]:
!git clone -q https://github.com/GinnyXiao/OpenWorldSAM.git
!pip install -q torch==2.5.1 torchvision==0.20.1
!pip install -q -r OpenWorldSAM/requirements.txt
!cd OpenWorldSAM/model/segment_anything_2 && python setup.py build_ext --inplace
!pip install -q --extra-index-url https://miropsota.github.io/torch_packages_builder "detectron2==0.6+2a420edpt2.5.0cu121"
!pip install -q "git+https://github.com/your-username/fishi.git"

In [ ]:
import os

os.makedirs("OpenWorldSAM/checkpoints", exist_ok=True)
!wget -q -O OpenWorldSAM/checkpoints/sam2_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

In [ ]:
from google.colab import drive

from fishi.evaluation import run
from fishi.preprocess import Identity, Patches, Rectify
from fishi.report import save_cell, to_csv
from fishi.segmentation.openworldsam import OpenWorldSam
from fishi.woodscape import classes
from fishi.woodscape.config import get_settings
from fishi.woodscape.dataset import WoodScapeDataset
from fishi.woodscape.splits import split_datasets

DRIVE_ROOT = "/content/drive/MyDrive/fishi"
DATA_DIRECTORY = f"{DRIVE_ROOT}/woodscape"
CACHE_DIRECTORY = f"{DRIVE_ROOT}/cache"
METRICS_DIRECTORY = f"{DRIVE_ROOT}/metrics"

OWS_REPO = "/content/OpenWorldSAM"
OWS_CONFIG = f"{OWS_REPO}/configs/your-config.yaml"
OWS_WEIGHTS = f"{OWS_REPO}/checkpoints/model.pth"

drive.mount("/content/drive")
settings = get_settings(data_directory=DATA_DIRECTORY)
test = split_datasets(WoodScapeDataset(settings))["test"]
print("test samples:", len(test))

In [ ]:
pipeline = OpenWorldSam(config_file=OWS_CONFIG, weights=OWS_WEIGHTS, repo_path=OWS_REPO)
for processor in (Identity(), Rectify(), Patches()):
    metrics = run(
        processor, pipeline, test, classes.PROMPTS,
        len(classes.CLASS_NAMES), cache_directory=CACHE_DIRECTORY,
    )
    save_cell(metrics, classes.CLASS_NAMES, pipeline.name, processor.name, METRICS_DIRECTORY)
    print(pipeline.name, processor.name, "mIoU", round(float(metrics["miou"]), 4))

to_csv(METRICS_DIRECTORY, f"{DRIVE_ROOT}/metrics.csv")